# Tempo até a reversa após a entrega ao cliente

**Pergunta de negócio:** depois que o pedido chega na casa do cliente, quanto tempo ele leva para *abrir uma reversa* (troca **ou** devolução)?

## Metodologia (rastreável)

| Elemento | Fonte | Campo |
|---|---|---|
| Chegada na casa do cliente | `insider-lake-sensitive.integrated_br.shippings_br` | `delivered_date` (DATETIME) — 1ª entrega por pedido (`MIN`) |
| Abertura da reversa | `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br` | `created_at` (TIMESTAMP) → convertido p/ `America/Sao_Paulo` |
| Tipo de reversa | idem Troquecommerce | `reverse_type` (Troca · Devolução · Troca e devolução · Sem Reembolso) |
| Pedidos válidos (filtro T&D) | `insider-data-lake.business.insider_orders` | `paid`, `is_cancelled = FALSE`, exclusão cupons TF-/TFIN/IR/Item errado, lojas `insider-world` + `insider-store-loja` |

**Chave de join:** `order_name` (formato `IN-XXXXXXX`, idêntico nas três tabelas — validado 2026-07-27).

**Métrica:** `dias_ate_reversa = created_at(BR) − delivered_date`, em dias fracionários. Grão = **uma linha por reversa** (`order_name × id_reversa`, dedup por `updated_at DESC`).

**Janela:** entregas nos **últimos 12 meses** (âncora = mês de entrega).

### Limitações declaradas
- **Censura à direita:** meses de entrega recentes ainda não tiveram tempo de acumular reversas tardias — a mediana/volume dos últimos ~30 dias é subestimada. Sinalizado nos gráficos de coorte.
- **Reversas antes da entrega (`dias < 0`, ~3–4%):** ruído de `delivered_date` (multi-volume, data de entrega imprecisa) ou reversa atrelada a outra remessa. Excluídas das estatísticas de tempo e reportadas à parte.
- Fonte da reversa é a **plataforma Troquecommerce** (quando o cliente *abriu* a reversa) — não confundir com `order_refunds_br` (evento financeiro no Shopify, universo maior).


## 0. Setup e parâmetros

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ID = "insider-data-lake"
client = bigquery.Client(project=PROJECT_ID)

# Janela: últimos 12 meses a partir de hoje (âncora = data de entrega)
HOJE = pd.Timestamp.today().normalize()
REF_DATE = (HOJE - pd.DateOffset(months=12)).date().isoformat()
DATA_TAG = HOJE.strftime("%Y%m%d")
OUT_DIR = "../../outputs"

print(f"Conectado a {PROJECT_ID}")

print(f"Janela de entregas: delivered_date >= {REF_DATE}  (hoje = {HOJE.date()})")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Conectado a insider-data-lake
Janela de entregas: delivered_date >= 2025-08-03  (hoje = 2026-08-03)


# Parte A — Tempo ENTREGA → reversa

*Do momento em que o pedido chega na casa do cliente (`delivered_date`) até a abertura da reversa.*

## 1. Extração — reversa × entrega (grão: uma linha por reversa)

In [2]:
query = f"""
WITH orders_validos AS (
  SELECT DISTINCT order_name
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_status = 'paid' AND is_cancelled = FALSE
    AND (coupon_code IS NULL OR (
      NOT STARTS_WITH(coupon_code, 'TF-')  AND NOT STARTS_WITH(coupon_code, 'TFIN')
      AND NOT STARTS_WITH(coupon_code, 'IR') AND NOT coupon_code LIKE '%Item errado%'))
    AND store IN ('shopify_insider-world', 'shopify_insider-store-loja')
),

-- 1ª entrega ao cliente por pedido
entrega AS (
  SELECT order_name, MIN(delivered_date) AS delivered_date
  FROM `insider-lake-sensitive.integrated_br.shippings_br`
  WHERE delivered_date IS NOT NULL
  GROUP BY order_name
),

-- reversa deduplicada por (order_name, id_reversa) via updated_at DESC
reversa AS (
  SELECT order_name, id_reversa, created_at, reverse_type
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND created_at IS NOT NULL AND id_reversa IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_name, id_reversa ORDER BY updated_at DESC) = 1
)

SELECT
  r.order_name,
  r.id_reversa,
  r.reverse_type,
  e.delivered_date,
  DATETIME(r.created_at, 'America/Sao_Paulo')                                             AS reversa_criada_em,
  DATETIME_DIFF(DATETIME(r.created_at, 'America/Sao_Paulo'), e.delivered_date, HOUR)/24.0  AS dias_ate_reversa
FROM reversa r
JOIN entrega e USING (order_name)
JOIN orders_validos o USING (order_name)
WHERE e.delivered_date >= '{REF_DATE}'
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)
print(f"{len(df):,} reversas casadas com entrega (pedidos válidos, últimos 12 meses)")
df.head()


112,932 reversas casadas com entrega (pedidos válidos, últimos 12 meses)


,order_name,id_reversa,reverse_type,delivered_date,reversa_criada_em,dias_ate_reversa
0,IN-2615291-SN,bdc181bb-6592-46e0-b8cc-eaff1dc71aff,Troca,2025-08-23 11:51:56,2025-09-22 14:32:02.019034,30.12
1,IN-2620464,f17cbd9c-e121-4ccb-a5bc-521776bfbdd4,Troca,2025-08-07 09:04:08,2025-08-09 14:43:08.366470,2.21
2,IN-2629126,f6b702d0-b633-4aef-a978-80677fe1acef,Troca,2025-08-07 18:15:53,2025-08-17 12:10:39.983526,9.75
3,IN-2629451,11ff26a1-5f36-471f-a8a2-ffbb897b38ce,Troca,2025-08-04 10:05:16,2025-08-10 19:26:57.655128,6.38
4,IN-2630048,45f89499-0487-49af-b0ec-157716051335,Troca,2025-08-09 10:52:12,2025-08-15 23:36:17.507896,6.54


## 2. Qualidade dos dados e enriquecimento

Excluímos `dias_ate_reversa < 0` das estatísticas de tempo (ruído de `delivered_date`) e criamos: mês de entrega, tipo simplificado e bucket de tempo.

In [3]:
# Enriquecimento
df["mes_entrega"] = pd.to_datetime(df["delivered_date"]).dt.to_period("M").astype(str)

# tipo simplificado (Troca / Devolução / Mista / Sem reembolso)
mapa_tipo = {
    "Troca": "Troca",
    "Devolução": "Devolução",
    "Troca e devolução": "Mista",
    "Sem Reembolso": "Sem reembolso",
}
df["tipo"] = df["reverse_type"].map(mapa_tipo).fillna("Outros")

# buckets de tempo
bins = [0, 2, 7, 15, 30, np.inf]
labels = ["0–2d", "3–7d", "8–15d", "16–30d", "31d+"]
df["bucket"] = pd.cut(df["dias_ate_reversa"], bins=bins, labels=labels, include_lowest=True)

# QA
n_total = len(df)
n_neg = int((df["dias_ate_reversa"] < 0).sum())
print(f"Total de reversas casadas : {n_total:,}")
print(f"Reversas antes da entrega : {n_neg:,}  ({n_neg/n_total:.1%}) -> excluídas do tempo")

# base válida para estatísticas de tempo
dfv = df[df["dias_ate_reversa"] >= 0].copy()
print(f"Base válida (dias >= 0)   : {len(dfv):,}")
df["reverse_type"].value_counts()


Total de reversas casadas : 112,932
Reversas antes da entrega : 4,133  (3.7%) -> excluídas do tempo
Base válida (dias >= 0)   : 108,799


reverse_type
Troca                99262
Devolução            12127
Sem Reembolso         1236
Troca e devolução      307
Name: count, dtype: int64

## 3. Distribuição geral — quantos dias até abrir a reversa

In [4]:
pcts = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
resumo_geral = dfv["dias_ate_reversa"].describe(percentiles=pcts).round(2)
print(resumo_geral)

# marcos práticos
for d in [1, 2, 7, 15, 30]:
    frac = (dfv["dias_ate_reversa"] <= d).mean()
    print(f"% das reversas abertas em até {d:>2}d após a entrega: {frac:.1%}")

# histograma interativo (cap visual em 60d para leitura)
cap = dfv["dias_ate_reversa"].clip(upper=60)
med = dfv["dias_ate_reversa"].median()

fig = px.histogram(cap, nbins=60, title="Dias entre entrega e abertura da reversa (cap visual 60d)",
                    labels={"value": "dias até a reversa"}, color_discrete_sequence=["#2b6cb0"])
fig.add_vline(x=med, line_dash="dash", line_color="#e53e3e",
              annotation_text=f"mediana = {med:.1f}d", annotation_position="top right")
fig.update_layout(xaxis_title="dias até a reversa", yaxis_title="nº de reversas", showlegend=False,
                   bargap=0.02, height=450)

fig.show()

count   108,799.00
mean          6.03
std          13.61
min           0.00
10%           0.12
25%           0.54
50%           2.12
75%           6.04
90%          13.79
95%          21.42
max         309.83
Name: dias_ate_reversa, dtype: float64
% das reversas abertas em até  1d após a entrega: 35.6%
% das reversas abertas em até  2d após a entrega: 48.6%
% das reversas abertas em até  7d após a entrega: 78.9%
% das reversas abertas em até 15d após a entrega: 91.2%
% das reversas abertas em até 30d após a entrega: 96.9%


## 3.1 Pareto — dias até a reversa vs. % acumulado coberto

A partir de quantos dias após a entrega já cobrimos 80% das reversas?


In [5]:
MAX_DIAS_PARETO = 60  # janela de leitura do pareto (dias)

dias_int = dfv["dias_ate_reversa"].clip(lower=0).apply(np.floor).astype(int)
contagem_dia = dias_int.value_counts().sort_index()
contagem_dia = contagem_dia.reindex(range(0, contagem_dia.index.max() + 1), fill_value=0)

pareto = contagem_dia.reset_index()
pareto.columns = ["dia", "n_reversas"]
pareto["pct_acumulado"] = pareto["n_reversas"].cumsum() / pareto["n_reversas"].sum() * 100

# dia em que se atinge >=80% (e outros marcos úteis)
marcos_pareto = {}
for alvo in [50, 80, 90, 95]:
    linha = pareto[pareto["pct_acumulado"] >= alvo].iloc[0]
    marcos_pareto[alvo] = int(linha["dia"])
    print(f">= {alvo}% das reversas cobertas a partir de {int(linha['dia'])} dias após a entrega")

dia_80 = marcos_pareto[80]

pareto_plot = pareto[pareto["dia"] <= MAX_DIAS_PARETO].copy()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=pareto_plot["dia"], y=pareto_plot["n_reversas"], name="nº reversas (dia)",
                      marker_color="#2b6cb0", opacity=0.6), secondary_y=False)
fig.add_trace(go.Scatter(x=pareto_plot["dia"], y=pareto_plot["pct_acumulado"], name="% acumulado",
                          mode="lines+markers", line=dict(color="#e53e3e")), secondary_y=True)
fig.add_hline(y=80, line_dash="dash", line_color="#e53e3e", secondary_y=True,
              annotation_text="80%", annotation_position="top left")
fig.add_vline(x=dia_80, line_dash="dot", line_color="#38a169",
              annotation_text=f"dia {dia_80}", annotation_position="top")
fig.update_yaxes(title_text="nº de reversas no dia", secondary_y=False)
fig.update_yaxes(title_text="% acumulado", range=[0, 100], secondary_y=True)
fig.update_xaxes(title_text="dias até a reversa (após entrega)")
fig.update_layout(title=f"Pareto — cobertura acumulada de reversas por dia (80% atingido no dia {dia_80})",
                   height=450)
fig.show()


>= 50% das reversas cobertas a partir de 2 dias após a entrega
>= 80% das reversas cobertas a partir de 7 dias após a entrega
>= 90% das reversas cobertas a partir de 13 dias após a entrega
>= 95% das reversas cobertas a partir de 21 dias após a entrega


## 4. Troca vs Devolução

In [6]:
stats_tipo = (dfv.groupby("tipo")["dias_ate_reversa"]
    .agg(n="count",
         mediana="median",
         media="mean",
         p90=lambda s: s.quantile(0.90),
         pct_ate_7d=lambda s: (s <= 7).mean())
    .sort_values("n", ascending=False)
    .round(2))
display(stats_tipo)

# boxplot interativo (troca vs devolução)
foco = dfv[dfv["tipo"].isin(["Troca", "Devolução"])].copy()
foco["dias_cap"] = foco["dias_ate_reversa"].clip(upper=60)

fig = px.box(foco, x="dias_cap", y="tipo", orientation="h", points=False,
             title="Distribuição de dias até a reversa — Troca vs Devolução (cap 60d)",
             labels={"dias_cap": "dias até a reversa", "tipo": ""},
             color="tipo", color_discrete_sequence=["#2b6cb0", "#e53e3e"])
fig.update_layout(showlegend=False, height=400)

fig.show()

,n,mediana,media,p90,pct_ate_7d
tipo,,,,,
Troca,95619,2.12,6.24,14.21,0.78
Devolução,11697,2.04,4.13,9.00,0.86
Sem reembolso,1190,2.79,8.81,18.98,0.73
Mista,293,2.12,3.69,9.36,0.85


## 5. Buckets de tempo (0–2d · 3–7d · 8–15d · 16–30d · 31d+)

In [7]:
# distribuição geral por bucket
dist_bucket = (dfv["bucket"].value_counts(normalize=True).reindex(labels) * 100).round(1)
tab_bucket = pd.DataFrame({"pct_%": dist_bucket, "n": dfv["bucket"].value_counts().reindex(labels)})
display(tab_bucket)

# crosstab % por tipo (Troca / Devolução)
ct = pd.crosstab(dfv["tipo"], dfv["bucket"], normalize="index")[labels] * 100
ct = ct.loc[[t for t in ["Troca", "Devolução", "Mista", "Sem reembolso"] if t in ct.index]].round(1)
display(ct)

# barra empilhada interativa
ct_long = ct.reset_index().melt(id_vars="tipo", var_name="bucket", value_name="pct")
fig = px.bar(ct_long, x="pct", y="tipo", color="bucket", orientation="h",
             title="Composição dos buckets de tempo por tipo de reversa (%)",
             labels={"pct": "% das reversas do tipo", "tipo": "", "bucket": "bucket"},
             category_orders={"bucket": labels}, color_discrete_sequence=px.colors.sequential.Blues_r,
             text=ct_long["pct"].map(lambda v: f"{v:.1f}%"))

fig.update_layout(barmode="stack", height=400)
fig.show()

,pct_%,n
bucket,,
0–2d,48.60,52853
3–7d,30.30,32940
8–15d,12.30,13402
16–30d,5.70,6244
31d+,3.10,3360


bucket,0–2d,3–7d,8–15d,16–30d,31d+
tipo,,,,,
Troca,48.50,29.60,12.60,6.10,3.30
Devolução,49.90,35.90,10.00,3.10,1.10
Mista,48.80,36.20,10.90,4.10,0.00
Sem reembolso,43.30,29.30,14.80,6.20,6.40


## 6. Coorte mensal (por mês de entrega)

⚠️ **Censura à direita:** os meses mais recentes ainda não tiveram tempo de acumular reversas tardias — mediana e volume subestimados no fim da série.

In [8]:
coorte = (dfv.groupby("mes_entrega")
    .agg(n_reversas=("id_reversa", "count"),
         mediana_dias=("dias_ate_reversa", "median"),
         pct_ate_7d=("dias_ate_reversa", lambda s: (s <= 7).mean() * 100),
         pct_ate_15d=("dias_ate_reversa", lambda s: (s <= 15).mean() * 100))
    .round(2))
display(coorte)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=coorte.index, y=coorte["n_reversas"], name="nº reversas",
                      marker_color="#718096", opacity=0.35), secondary_y=True)
fig.add_trace(go.Scatter(x=coorte.index, y=coorte["mediana_dias"], name="mediana (dias)",
                          mode="lines+markers", marker=dict(color="#2b6cb0"), line=dict(color="#2b6cb0")),
              secondary_y=False)
fig.update_yaxes(title_text="mediana de dias até a reversa", secondary_y=False)
fig.update_yaxes(title_text="nº de reversas", secondary_y=True)
fig.update_xaxes(tickangle=45)
fig.update_layout(title="Coorte por mês de entrega — mediana de dias até a reversa e volume", height=450)

fig.show()

,n_reversas,mediana_dias,pct_ate_7d,pct_ate_15d
mes_entrega,,,,
2025-08,7452,2.33,79.07,91.17
2025-09,7810,2.08,79.80,91.18
2025-10,8386,2.04,80.40,91.82
2025-11,13488,2.00,80.12,91.20
2025-12,16099,3.54,67.20,84.03
2026-01,8173,2.04,79.51,91.21
2026-02,7496,2.00,79.02,91.74
2026-03,13904,1.88,82.13,92.98
2026-04,8181,2.08,79.93,92.63


# Parte B — Tempo COMPRA → reversa

*Da **data da compra** (`processed_at` → `America/Sao_Paulo`) até a abertura da reversa.*
População = todas as reversas válidas com compra casada por `order_name`; janela = **compras dos últimos 12 meses**. Validado (2026-07-27): **0 reversas antes da compra** e **99,4%** também têm entrega casada — as duas populações praticamente coincidem.

## 7. Extração — compra × reversa

In [9]:
query_compra = f"""
WITH compras AS (
  SELECT DISTINCT
    order_name,
    DATE(TIMESTAMP(processed_at), 'America/Sao_Paulo') AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_status = 'paid' AND is_cancelled = FALSE
    AND (coupon_code IS NULL OR (
      NOT STARTS_WITH(coupon_code, 'TF-')  AND NOT STARTS_WITH(coupon_code, 'TFIN')
      AND NOT STARTS_WITH(coupon_code, 'IR') AND NOT coupon_code LIKE '%Item errado%'))
    AND order_name IS NOT NULL AND processed_at IS NOT NULL
    AND store IN ('shopify_insider-world', 'shopify_insider-store-loja')
),

reversa AS (
  SELECT order_name, id_reversa, created_at, reverse_type
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND created_at IS NOT NULL AND id_reversa IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_name, id_reversa ORDER BY updated_at DESC) = 1
),

entrega AS (
  SELECT order_name, MIN(delivered_date) AS delivered_date
  FROM `insider-lake-sensitive.integrated_br.shippings_br`
  WHERE delivered_date IS NOT NULL
  GROUP BY order_name
)

SELECT
  r.order_name,
  r.id_reversa,
  r.reverse_type,
  c.data_compra,
  DATE(e.delivered_date)                                             AS data_entrega,
  DATE(r.created_at, 'America/Sao_Paulo')                            AS data_reversa,
  DATE_DIFF(DATE(r.created_at, 'America/Sao_Paulo'), c.data_compra, DAY) AS dias_compra_ate_reversa
FROM reversa r
JOIN compras c USING (order_name)
LEFT JOIN entrega e USING (order_name)
WHERE c.data_compra >= '{REF_DATE}'
"""

df_compra = client.query(query_compra).to_dataframe(create_bqstorage_client=False)

# enriquecimento
df_compra["mes_compra"] = pd.to_datetime(df_compra["data_compra"]).dt.to_period("M").astype(str)
df_compra["tipo"] = df_compra["reverse_type"].map(mapa_tipo).fillna("Outros")
bins_c = [0, 7, 15, 30, 60, np.inf]
labels_c = ["0–7d", "8–15d", "16–30d", "31–60d", "61d+"]
df_compra["bucket"] = pd.cut(df_compra["dias_compra_ate_reversa"], bins=bins_c, labels=labels_c, include_lowest=True)

n = len(df_compra); neg = int((df_compra["dias_compra_ate_reversa"] < 0).sum())
tem_ent = df_compra["data_entrega"].notna().mean()
print(f"Reversas com compra casada : {n:,}")
print(f"Negativas (reversa < compra): {neg:,}  ({neg/n:.2%})")
print(f"Tambem com entrega casada  : {tem_ent:.1%}")

dfc = df_compra[df_compra["dias_compra_ate_reversa"] >= 0].copy()
df_compra.head()


Reversas com compra casada : 111,333
Negativas (reversa < compra): 0  (0.00%)
Tambem com entrega casada  : 99.4%


,order_name,id_reversa,reverse_type,data_compra,data_entrega,data_reversa,dias_compra_ate_reversa,mes_compra,tipo,bucket
0,IN-2701995,37ba2a2f-c251-452b-9b0c-fa12b4e2def0,Sem Reembolso,2025-08-12,2025-08-19,2025-10-15,64,2025-08,Sem reembolso,61d+
1,IN-3255245-SN,ea9729a2-1e80-4f58-b74d-c02cb9e2b571,Troca,2025-11-28,2025-12-06,2026-01-12,45,2025-11,Troca,31–60d
2,IN-3302182-SN,a0daaf7a-cd39-441e-8cc3-bde7140d1509,Troca,2025-11-30,2025-12-10,2026-05-06,157,2025-11,Troca,61d+
3,IN-3366749-ENTR-0126,863f0dab-b7a7-4585-9cd8-99a2a333e4d9,Troca,2025-12-10,2026-02-06,2026-02-10,62,2025-12,Troca,61d+
4,IN-3469231,063f6c5d-3b2c-4c34-b22b-49dc3bd672fc,Troca,2025-12-30,2026-01-13,2026-02-02,34,2025-12,Troca,31–60d


## 8. Distribuição e tipo (compra → reversa)

In [10]:
pcts = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
print(dfc["dias_compra_ate_reversa"].describe(percentiles=pcts).round(2))
for d in [7, 15, 30, 60]:
    print(f"% das reversas abertas em até {d:>2}d após a compra: {(dfc['dias_compra_ate_reversa'] <= d).mean():.1%}")

# histograma interativo (cap visual em 90d para leitura)
cap = dfc["dias_compra_ate_reversa"].clip(upper=90)
med = dfc["dias_compra_ate_reversa"].median()

fig = px.histogram(cap, nbins=60, title="Dias entre compra e abertura da reversa (cap visual 90d)",
                    labels={"value": "dias após a compra"}, color_discrete_sequence=["#2f855a"])
fig.add_vline(x=med, line_dash="dash", line_color="#e53e3e",
              annotation_text=f"mediana = {med:.0f}d", annotation_position="top right")
fig.update_layout(xaxis_title="dias após a compra", yaxis_title="nº de reversas", showlegend=False,
                   bargap=0.02, height=450)
fig.show()

stats_tipo_compra = (dfc.groupby("tipo")["dias_compra_ate_reversa"]
    .agg(n="count", mediana="median", media="mean",
         p90=lambda s: s.quantile(0.90),
         pct_ate_15d=lambda s: (s <= 15).mean())
    .sort_values("n", ascending=False).round(2))
display(stats_tipo_compra)


count   111,333.00
mean         12.88
std          16.21
min           0.00
10%           4.00
25%           6.00
50%           8.00
75%          14.00
90%          23.00
95%          35.00
max         314.00
Name: dias_compra_ate_reversa, dtype: Float64
% das reversas abertas em até  7d após a compra: 42.4%
% das reversas abertas em até 15d após a compra: 79.9%
% das reversas abertas em até 30d após a compra: 94.0%
% das reversas abertas em até 60d após a compra: 97.8%


,n,mediana,media,p90,pct_ate_15d
tipo,,,,,
Troca,97753,9.00,13.00,24,0.79
Devolução,12069,8.00,11.86,20,0.84
Sem reembolso,1203,9.00,14.66,26,0.78
Mista,308,9.00,10.82,18,0.86


## 8.1 Pareto — dias até a reversa (compra) vs. % acumulado coberto

A partir de quantos dias após a compra já cobrimos 80% das reversas?


In [11]:
MAX_DIAS_PARETO_COMPRA = 90  # janela de leitura do pareto (dias)

dias_int_c = dfc["dias_compra_ate_reversa"].clip(lower=0).apply(np.floor).astype(int)
contagem_dia_c = dias_int_c.value_counts().sort_index()
contagem_dia_c = contagem_dia_c.reindex(range(0, contagem_dia_c.index.max() + 1), fill_value=0)

pareto_compra = contagem_dia_c.reset_index()
pareto_compra.columns = ["dia", "n_reversas"]
pareto_compra["pct_acumulado"] = pareto_compra["n_reversas"].cumsum() / pareto_compra["n_reversas"].sum() * 100

# dia em que se atinge >=80% (e outros marcos úteis)
marcos_pareto_compra = {}
for alvo in [50, 80, 90, 95]:
    linha = pareto_compra[pareto_compra["pct_acumulado"] >= alvo].iloc[0]
    marcos_pareto_compra[alvo] = int(linha["dia"])
    print(f">= {alvo}% das reversas cobertas a partir de {int(linha['dia'])} dias após a compra")

dia_80_compra = marcos_pareto_compra[80]

pareto_compra_plot = pareto_compra[pareto_compra["dia"] <= MAX_DIAS_PARETO_COMPRA].copy()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=pareto_compra_plot["dia"], y=pareto_compra_plot["n_reversas"], name="nº reversas (dia)",
                      marker_color="#2f855a", opacity=0.6), secondary_y=False)
fig.add_trace(go.Scatter(x=pareto_compra_plot["dia"], y=pareto_compra_plot["pct_acumulado"], name="% acumulado",
                          mode="lines+markers", line=dict(color="#e53e3e")), secondary_y=True)
fig.add_hline(y=80, line_dash="dash", line_color="#e53e3e", secondary_y=True,
              annotation_text="80%", annotation_position="top left")
fig.add_vline(x=dia_80_compra, line_dash="dot", line_color="#2b6cb0",
              annotation_text=f"dia {dia_80_compra}", annotation_position="top")
fig.update_yaxes(title_text="nº de reversas no dia", secondary_y=False)
fig.update_yaxes(title_text="% acumulado", range=[0, 100], secondary_y=True)
fig.update_xaxes(title_text="dias até a reversa (após compra)")
fig.update_layout(title=f"Pareto — cobertura acumulada de reversas por dia após a compra (80% atingido no dia {dia_80_compra})",
                   height=450)
fig.show()


>= 50% das reversas cobertas a partir de 8 dias após a compra
>= 80% das reversas cobertas a partir de 16 dias após a compra
>= 90% das reversas cobertas a partir de 23 dias após a compra
>= 95% das reversas cobertas a partir de 35 dias após a compra


## 9. Buckets e coorte mensal (por mês de compra)

⚠️ **Censura à direita:** meses de compra recentes ainda não tiveram tempo de gerar reversas tardias — subestimados no fim da série.

In [12]:
tab_bucket_compra = pd.DataFrame({
    "pct_%": (dfc["bucket"].value_counts(normalize=True).reindex(labels_c) * 100).round(1),
    "n": dfc["bucket"].value_counts().reindex(labels_c),
})
display(tab_bucket_compra)

coorte_compra = (dfc.groupby("mes_compra")
    .agg(n_reversas=("id_reversa", "count"),
         mediana_dias=("dias_compra_ate_reversa", "median"),
         pct_ate_15d=("dias_compra_ate_reversa", lambda s: (s <= 15).mean() * 100),
         pct_ate_30d=("dias_compra_ate_reversa", lambda s: (s <= 30).mean() * 100))
    .round(2))
display(coorte_compra)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=coorte_compra.index, y=coorte_compra["n_reversas"], name="nº reversas",
                      marker_color="#718096", opacity=0.35), secondary_y=True)
fig.add_trace(go.Scatter(x=coorte_compra.index, y=coorte_compra["mediana_dias"], name="mediana (dias)",
                          mode="lines+markers", marker=dict(color="#2f855a"), line=dict(color="#2f855a")),
              secondary_y=False)
fig.update_yaxes(title_text="mediana dias compra→reversa", secondary_y=False)
fig.update_yaxes(title_text="nº de reversas", secondary_y=True)
fig.update_xaxes(tickangle=45)
fig.update_layout(title="Coorte por mês de compra — mediana compra→reversa e volume", height=450)
fig.show()


,pct_%,n
bucket,,
0–7d,42.40,47184
8–15d,37.60,41818
16–30d,14.00,15633
31–60d,3.90,4294
61d+,2.20,2404


,n_reversas,mediana_dias,pct_ate_15d,pct_ate_30d
mes_compra,,,,
2025-08,7630,9.00,78.74,92.15
2025-09,7720,8.00,80.82,92.46
2025-10,9344,9.00,81.54,94.30
2025-11,19751,10.00,74.99,91.85
2025-12,11464,12.00,64.10,90.47
2026-01,7813,7.00,82.96,93.31
2026-02,8288,8.00,81.59,94.91
2026-03,14812,7.00,86.15,96.73
2026-04,7281,8.00,82.17,94.93


## 10. Decomposição compra → entrega → reversa

Nas reversas com entrega casada, o tempo total compra→reversa se divide em **envio** (compra→entrega) e **decisão** (entrega→reversa). Medianas não somam exatamente (distribuições diferentes).

In [13]:
dec = df_compra[df_compra["data_entrega"].notna()].copy()
dec["dias_compra_entrega"] = (pd.to_datetime(dec["data_entrega"]) - pd.to_datetime(dec["data_compra"])).dt.days
dec["dias_entrega_reversa"] = (pd.to_datetime(dec["data_reversa"]) - pd.to_datetime(dec["data_entrega"])).dt.days
dec_v = dec[(dec["dias_compra_entrega"] >= 0) & (dec["dias_entrega_reversa"] >= 0) & (dec["dias_compra_ate_reversa"] >= 0)]

tab_dec = pd.DataFrame({
    "etapa": ["compra -> entrega (envio)", "entrega -> reversa (decisao)", "compra -> reversa (total)"],
    "mediana_dias": [dec_v["dias_compra_entrega"].median(),
                     dec_v["dias_entrega_reversa"].median(),
                     dec_v["dias_compra_ate_reversa"].median()],
    "media_dias": [dec_v["dias_compra_entrega"].mean(),
                   dec_v["dias_entrega_reversa"].mean(),
                   dec_v["dias_compra_ate_reversa"].mean()],
}).round(1)
display(tab_dec)
print(f"Base decomposicao (3 marcos casados, nao-negativos): {len(dec_v):,}")


,etapa,mediana_dias,media_dias
0,compra -> entrega (envio),5.00,7.00
1,entrega -> reversa (decisao),2.00,5.80
2,compra -> reversa (total),8.00,12.90


Base decomposicao (3 marcos casados, nao-negativos): 109,080


# Parte C — OKR de Troca & Devolução (Source of Truth · COHORT)

> **Fonte da verdade (SoT):** métrica **por coorte de compra**. A reversa é atribuída ao **mês/semana da compra** do pedido, via join `(order_name, sku)`. Numerador **lifetime** (todas as reversas do item, sem corte por data).
> - **Numerador:** `SUM(return_quantity)` de reversas dos itens comprados no período (dedup `order_name,id_reversa,sku`).
> - **Denominador:** `SUM(quantity)` de itens comprados no período (`paid`/filtros de cupom/lojas).
> - `T&D% = reversas_do_cohort ÷ vendas_do_cohort × 100`.

**Definições + maturação:**
- **KR mensal (MBR):** headline = **mês passado (M-1)**, calculado no **dia 15 de M+1** (buffer de maturação do cohort). Meses recentes = `em maturação`.
- **HM semanal:** cohort de compra **[D-21, D-15]** = semana **W-2** (reportável). Semanas mais recentes ainda estão maturando (poucas reversas) → subestimadas.

## 11. Extração diária do cohort (base p/ KR mensal e HM semanal)

In [14]:
query_cohort = """
WITH compras_filtradas AS (
  SELECT DISTINCT order_id, order_name, DATE(TIMESTAMP(processed_at),'America/Sao_Paulo') AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_status='paid' AND is_cancelled=FALSE
    AND (coupon_code IS NULL OR (NOT STARTS_WITH(coupon_code,'TF-') AND NOT STARTS_WITH(coupon_code,'TFIN')
      AND NOT STARTS_WITH(coupon_code,'IR') AND NOT coupon_code LIKE '%Item errado%'))
    AND order_name IS NOT NULL AND processed_at IS NOT NULL
    AND store IN ('shopify_insider-world','shopify_insider-store-loja')),
itens_comprados AS (
  SELECT c.data_compra, c.order_name, i.sku, SUM(i.quantity) AS qt_comprados
  FROM compras_filtradas c JOIN `insider-data-lake.business.insider_order_items` i ON c.order_id=i.order_id
  WHERE i.sku IS NOT NULL AND c.data_compra >= '2025-01-01' GROUP BY 1,2,3),
reversas_por_item AS (
  SELECT order_name, sku, SUM(return_quantity) AS qt_revertidos
  FROM (SELECT DISTINCT order_name, id_reversa, sku, SAFE_CAST(return_quantity AS FLOAT64) AS return_quantity
        FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
        WHERE status<>'Cancelado' AND order_name IS NOT NULL AND sku IS NOT NULL AND id_reversa IS NOT NULL)
  GROUP BY 1,2)
SELECT ic.data_compra AS dia, SUM(ic.qt_comprados) AS vendas, SUM(COALESCE(r.qt_revertidos,0)) AS reversas
FROM itens_comprados ic LEFT JOIN reversas_por_item r ON ic.order_name=r.order_name AND ic.sku=r.sku
GROUP BY 1 ORDER BY 1
"""
df_cohort = client.query(query_cohort).to_dataframe(create_bqstorage_client=False)
df_cohort["dia"] = pd.to_datetime(df_cohort["dia"])
HOJE_D = pd.Timestamp.today().normalize()
print(f"{len(df_cohort)} dias | vendas {df_cohort['vendas'].sum():,.0f} | reversas {df_cohort['reversas'].sum():,.0f}")
df_cohort.tail()


580 dias | vendas 4,934,095 | reversas 237,068


,dia,vendas,reversas
575,2026-07-30,7345,10.00
576,2026-07-31,8701,0.00
577,2026-08-01,7203,0.00
578,2026-08-02,9070,0.00
579,2026-08-03,226,0.00


## 12. KR mensal (MBR) — headline do "mês passado"

`T&D% = reversas do cohort do mês ÷ vendas do mês`. Headline = último mês cuja maturação (dia 15 de M+1) já passou; demais = `em maturação`.

In [15]:
dm = df_cohort[df_cohort["dia"] >= "2025-01-01"].copy()
dm["mes"] = dm["dia"].dt.to_period("M")
kr = dm.groupby("mes").agg(vendas=("vendas","sum"), reversas=("reversas","sum")).reset_index()
kr["td_pct"] = (kr["reversas"]/kr["vendas"]*100).round(2)
kr["maduro_em"] = (kr["mes"].dt.to_timestamp()+pd.offsets.MonthBegin(1))+pd.Timedelta(days=14)  # dia 15 de M+1
kr["status"] = np.where(HOJE_D >= kr["maduro_em"], "oficial (maduro)", "em maturação")
kr["mes"] = kr["mes"].astype(str)
display(kr[["mes","vendas","reversas","td_pct","status","maduro_em"]].tail(12))

oficiais = kr[kr["status"].str.startswith("oficial")]
head = oficiais.iloc[-1]
prox = kr[kr["status"]=="em maturação"].iloc[0] if (kr["status"]=="em maturação").any() else None
print(f"KR VIGENTE (mês passado maduro): {head['mes']} = {head['td_pct']:.2f}%")
if prox is not None:
    print(f"Próxima leitura: {prox['mes']} (provisório {prox['td_pct']:.2f}%) — oficial em {prox['maduro_em'].date()}")

fig = px.bar(oficiais.tail(12), x="mes", y="td_pct", text="td_pct",
             title=f"KR mensal (SoT cohort) — T&D% oficial · headline {head['mes']} = {head['td_pct']:.2f}%",
             labels={"mes":"", "td_pct":"T&D (%)"}, color_discrete_sequence=["#2b6cb0"])
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(height=420)
fig.show()


,mes,vendas,reversas,td_pct,status,maduro_em
8,2025-09,266541,"12,621.00",4.74,oficial (maduro),2025-10-15
9,2025-10,283615,"15,137.00",5.34,oficial (maduro),2025-11-15
10,2025-11,630261,"32,830.00",5.21,oficial (maduro),2025-12-15
11,2025-12,333045,"17,068.00",5.12,oficial (maduro),2026-01-15
12,2026-01,217791,"11,838.00",5.44,oficial (maduro),2026-02-15
13,2026-02,228536,"13,164.00",5.76,oficial (maduro),2026-03-15
14,2026-03,376559,"23,666.00",6.28,oficial (maduro),2026-04-15
15,2026-04,244773,"12,031.00",4.92,oficial (maduro),2026-05-15
16,2026-05,195670,"9,858.00",5.04,oficial (maduro),2026-06-15
17,2026-06,185589,"9,240.00",4.98,oficial (maduro),2026-07-15


KR VIGENTE (mês passado maduro): 2026-06 = 4.98%
Próxima leitura: 2026-07 (provisório 4.34%) — oficial em 2026-08-15


## 13. HM semanal contínuo (cohort · W-2)

Série semanal por **semana de compra** (cohort lifetime). Reportável até **W-2 = [D-21, D-15]**; semanas mais recentes ainda maturando (subestimadas).

In [16]:
df_cohort["semana_ini"] = df_cohort["dia"].dt.to_period("W-SUN").dt.start_time
wk = (df_cohort.groupby("semana_ini").agg(vendas=("vendas","sum"), reversas=("reversas","sum"))
      .reset_index().sort_values("semana_ini"))
wk["td_pct"] = (wk["reversas"]/wk["vendas"]*100).round(2)
wk["media_movel_4s"] = wk["td_pct"].rolling(4).mean().round(2)
d21 = HOJE_D - pd.Timedelta(days=21)
w2_mon = d21 - pd.Timedelta(days=d21.weekday())   # semana [D-21, D-15]
wk["reportavel"] = wk["semana_ini"] <= w2_mon
wk["is_w2"] = wk["semana_ini"] == w2_mon
display(wk[["semana_ini","vendas","reversas","td_pct","media_movel_4s","reportavel"]].tail(8))
v = wk[wk["is_w2"]].iloc[0]
print(f"HM W-2 [{d21.date()}..{(HOJE_D-pd.Timedelta(days=15)).date()}] (semana {v['semana_ini'].date()}) = {v['td_pct']:.2f}%")

rep = wk[wk["reportavel"]]; imat = wk[~wk["reportavel"]]
fig = go.Figure()
if len(imat):
    fig.add_vrect(x0=imat["semana_ini"].min(), x1=wk["semana_ini"].max(), fillcolor="#f6ad55", opacity=0.12, line_width=0,
                  annotation_text="imaturo (> W-2)", annotation_position="top left")
fig.add_trace(go.Scatter(x=rep["semana_ini"], y=rep["td_pct"], mode="lines+markers",
                         name="T&D cohort semanal (≤ W-2)", line=dict(color="#2b6cb0")))
if len(imat):
    link = pd.concat([rep.tail(1), imat])
    fig.add_trace(go.Scatter(x=link["semana_ini"], y=link["td_pct"], mode="lines+markers",
                             name="ainda maturando (> W-2)", line=dict(color="#a0aec0", dash="dash")))
fig.add_trace(go.Scatter(x=rep["semana_ini"], y=rep["media_movel_4s"], mode="lines",
                         name="média móvel 4 sem", line=dict(color="#dd6b20", width=3)))
fig.add_vline(x=v["semana_ini"], line=dict(color="#38a169", dash="dot"))
fig.add_annotation(x=v["semana_ini"], y=v["td_pct"], text=f"W-2 (HM) {v['td_pct']:.2f}%",
                   showarrow=True, arrowhead=2, arrowcolor="#38a169", font=dict(color="#276749"))
fig.update_layout(title="T&D COHORT semanal (%) — reversas das compras da semana ÷ vendas da semana (SoT)",
                  xaxis_title="semana de compra (início)", yaxis_title="T&D (%)", height=470)
fig.show()


,semana_ini,vendas,reversas,td_pct,media_movel_4s,reportavel
76,2026-06-15,35018,"1,868.00",5.33,4.87,True
77,2026-06-22,31992,"1,764.00",5.51,5.06,True
78,2026-06-29,43702,"2,497.00",5.71,5.36,True
79,2026-07-06,44089,"2,402.00",5.45,5.50,True
80,2026-07-13,37857,"2,092.00",5.53,5.55,True
81,2026-07-20,38648,"1,565.00",4.05,5.18,False
82,2026-07-27,53151,290.00,0.55,3.90,False
83,2026-08-03,226,0.00,0.00,2.53,False


HM W-2 [2026-07-13..2026-07-19] (semana 2026-07-13) = 5.53%


## 14. Export dos CSVs


In [17]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

# --- Parte A: entrega -> reversa ---
df.to_csv(f"{OUT_DIR}/tempo_reversa_base_{DATA_TAG}.csv", index=False)
stats_tipo.to_csv(f"{OUT_DIR}/tempo_reversa_por_tipo_{DATA_TAG}.csv")
tab_bucket.to_csv(f"{OUT_DIR}/tempo_reversa_buckets_{DATA_TAG}.csv")
coorte.to_csv(f"{OUT_DIR}/tempo_reversa_coorte_mensal_{DATA_TAG}.csv")

# --- Parte B: compra -> reversa ---
df_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_base_{DATA_TAG}.csv", index=False)
stats_tipo_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_por_tipo_{DATA_TAG}.csv")
tab_bucket_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_buckets_{DATA_TAG}.csv")
coorte_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_coorte_mensal_{DATA_TAG}.csv")
tab_dec.to_csv(f"{OUT_DIR}/tempo_decomposicao_compra_entrega_reversa_{DATA_TAG}.csv", index=False)

# --- Parte C: OKR T&D (SoT cohort) ---
kr[["mes","vendas","reversas","td_pct","status"]].to_csv(f"{OUT_DIR}/okr_td_kr_mensal_{DATA_TAG}.csv", index=False)
wk.to_csv(f"{OUT_DIR}/okr_td_hm_semanal_{DATA_TAG}.csv", index=False)
df_cohort.to_csv(f"{OUT_DIR}/okr_td_cohort_diario_{DATA_TAG}.csv", index=False)
print("  Parte C: okr_td_{kr_mensal,hm_semanal,cohort_diario}")


  Parte C: okr_td_{kr_mensal,hm_semanal,cohort_diario}


## 15. Sumário executivo

*Execução: 2026-07-27 · entregas dos últimos 12 meses · pedidos válidos (filtros T&D) · fonte da reversa = Troquecommerce (`created_at`).*
*Base: 113.389 reversas casadas com entrega; 3,7% abertas antes da `delivered_date` (ruído) excluídas → 109.232 válidas.*

**1. A decisão de reverter é rápida e concentrada nas primeiras 48h.**
Mediana de **2,1 dias** entre a entrega e a abertura da reversa. **48,6%** das reversas são abertas em até 2 dias, **78,8%** em até 7 dias e **91,2%** em até 15 dias. A janela de intervenção (retenção/UX) é curtíssima.

**2. Troca domina o volume — ~8× mais que devolução.**
Troca = **96.174** reversas · Devolução = **11.514** (via Troquecommerce). A Insider canaliza a insatisfação para troca, não para saída de caixa via devolução.

**3. Devolução decide mais rápido e com cauda mais curta que troca.**
Devolução: mediana 2,0d, **p90 9,0d**, 86% em ≤7d. Troca: mediana 2,1d, **p90 14,2d**, 78% em ≤7d. Quem vai devolver bate o martelo cedo; a troca tem mais gente que "pensa mais" (experimenta, avalia grade/tamanho) e volta em 1–2 semanas.

**4. Distribuição por bucket:** 0–2d **48,6%** · 3–7d **30,2%** · 8–15d **12,4%** · 16–30d **5,7%** · 31d+ **3,1%**.

### Implicações e próximos passos
- **Operação/UX:** ação de retenção tem valor máximo nas **primeiras 48h pós-entrega** (grade alternativa, ajuste de tamanho, prova virtual). Depois de 7 dias, ~79% da reversa já foi disparada.
- **Aprofundar causa (moonshot):** cruzar `dias_ate_reversa` × `return_reason` × categoria para separar reversa por **fit/tamanho** (decisão rápida, 0–2d) de **defeito/qualidade** (tende a aparecer mais tarde) — muda a alavanca (curadoria de grade vs. QA de fornecedor).
- **Rigor estatístico:** os meses de entrega mais recentes têm **censura à direita**; para tendência limpa, reprocessar excluindo os últimos ~30 dias de entrega ou usar coorte fechada.


---

### Parte B — Compra → reversa (fecha o ciclo)

- **Mediana de 9 dias** entre a compra e a abertura da reversa (p25 6d · p90 23d). **0 reversas** ocorrem antes da compra (métrica limpa, sem exclusões).
- **O gargalo é o frete, não a indecisão:** decompondo, a mediana **compra→entrega (envio) é ~5 dias** e **entrega→reversa (decisão) é ~2 dias**. O cliente decide rápido; o relógio corre no transporte.
- **Alavanca dupla:** reduzir lead time de entrega antecipa (e pode reduzir) reversas por ansiedade/atraso; agir na experiência nas primeiras 48h pós-entrega ataca a janela de decisão. As duas frentes são complementares.

---

### Parte C — OKR de T&D (Source of Truth · COHORT) — execução 2026-08-03

- **Definição:** por **coorte de compra** — reversa atribuída ao mês/semana da compra (`order_name+sku`), numerador lifetime. `T&D% = reversas do cohort ÷ vendas do cohort`.
- **KR mensal (headline = mês passado maduro):** **jun/26 = 4,98%** (vigente). jul/26 (4,34% provisório) fica oficial em 15/ago. Pico do ano: **mar/26 = 6,28%**.
- **HM semanal (W-2 = [D-21..D-15] = semana 13/jul):** **5,53%**. Semanas > W-2 ainda maturando (subestimadas); ler tendência pela média móvel 4 sem.
